<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс BankAccount в C#, который будет представлять 
информацию об учетных записях в банке. На основе этого класса разработать 2-3 
производных класса, демонстрирующих принципы наследования и полиморфизма. 
В каждом из классов должны быть реализованы новые атрибуты и методы, а также 
переопределены некоторые методы базового класса для демонстрации 
полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [5]:
using System;
using System.Collections.Generic;

public abstract class BankAccount
{
    public string AccountNumber { get; set; }
    public decimal Balance { get; set; }
    public string AccountType { get; protected set; }
    public string Owner { get; set; }

    public BankAccount(string accNum, decimal balance, string type, string owner)
    {
        AccountNumber = accNum;
        Balance = balance;
        AccountType = type;
        Owner = owner;
    }

    public virtual string GetInfo()
    {
        return $"{AccountNumber,-10} | {Balance,12:C} | {AccountType,-15} | {Owner,-10}";
    }

    public virtual void Deposit(decimal amount)
    {
        if (amount > 0) Balance += amount;
    }

    public virtual void Deposit(decimal amount, string comment)
    {
        Deposit(amount);
        Console.WriteLine($"[Пополнение] Счет {AccountNumber}: {amount:C} ({comment})");
    }

    public virtual void Withdraw(decimal amount)
    {
        if (amount > 0 && amount <= Balance)
            Balance -= amount;
    }
}

public class SavingsAccount : BankAccount
{
    public decimal InterestRate { get; set; }

    public SavingsAccount(string accNum, decimal balance, string owner, decimal interest)
        : base(accNum, balance, "Сберегательный", owner)
    {
        InterestRate = interest;
    }

    public override void Deposit(decimal amount)
    {
        decimal bonus = amount * InterestRate;
        base.Deposit(amount + bonus);
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $" | Процент: {InterestRate:P}";
    }
}

public class CheckingAccount : BankAccount
{
    public decimal OverdraftLimit { get; set; }

    public CheckingAccount(string accNum, decimal balance, string owner, decimal overdraft)
        : base(accNum, balance, "Текущий", owner)
    {
        OverdraftLimit = overdraft;
    }

    public override void Withdraw(decimal amount)
    {
        if (amount > 0 && amount <= Balance + OverdraftLimit)
            Balance -= amount;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $" | Овердрафт: {OverdraftLimit:C}";
    }
}

public class InvestmentAccount : BankAccount
{
    public List<string> Assets { get; set; }

    public InvestmentAccount(string accNum, decimal balance, string owner, List<string> assets)
        : base(accNum, balance, "Инвестиционный", owner)
    {
        Assets = assets;
    }

    public void AddAsset(string asset)
    {
        Assets.Add(asset);
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $" | Активы: {string.Join(", ", Assets)}";
    }
}

public class AccountManager<T> where T : BankAccount
{
    private List<T> accounts = new List<T>();

    public void AddAccount(T account)
    {
        accounts.Add(account);
    }

    public void PrintAllAccounts()
    {
        Console.WriteLine(new string('-', 85));
        Console.WriteLine($"{"Номер счета",-10} | {"Баланс",12} | {"Тип счета",-15} | {"Владелец",-10} | Доп. информация");
        Console.WriteLine(new string('-', 85));
        foreach(var acc in accounts)
        {
            Console.WriteLine(acc.GetInfo());
        }
        Console.WriteLine(new string('-', 85));
    }
}

var savings = new SavingsAccount("SA001", 1000, "Иван", 0.05m);
var checking = new CheckingAccount("CH001", 500, "Петр", 200);
var investment = new InvestmentAccount("IN001", 10000, "Мария", new List<string>{ "Акции", "Облигации" });

savings.Deposit(100);
savings.Deposit(100, "Подарок");
checking.Withdraw(600);
investment.AddAsset("ETF");

var manager = new AccountManager<BankAccount>();
manager.AddAccount(savings);
manager.AddAccount(checking);
manager.AddAccount(investment);

Console.WriteLine();
manager.PrintAllAccounts();


[Пополнение] Счет SA001: ¤100.00 (Подарок)

-------------------------------------------------------------------------------------
Номер счета |       Баланс | Тип счета       | Владелец   | Доп. информация
-------------------------------------------------------------------------------------
SA001      |    ¤1,210.00 | Сберегательный  | Иван       | Процент: 5.000%
CH001      |     -¤100.00 | Текущий         | Петр       | Овердрафт: ¤200.00
IN001      |   ¤10,000.00 | Инвестиционный  | Мария      | Активы: Акции, Облигации, ETF
-------------------------------------------------------------------------------------
